# Regime-Stratified Sensitivity Analysis — Level 5L6 (isothermal)

Sensitivity of the L5L6 model (defective oxide L3 + defective metal L4 + surface kinetics L6)
**conditioned on the rate-limiting regime**, run **isothermally** at several temperatures.

## Regimes — surface / oxide / metal

`regime = argmax` of the flux-weighted resistance fractions
(`frac_surface`, `frac_oxide`, `frac_metal`), all three of which are **series** resistances.

> **Note — the parallel path is computed but not part of the label.**
> `calculate_full_model_flux_L346_v2` does model the defect bypass (pinhole / crack /
> grain-boundary, each with its own permeance), and `flux_intact` / `flux_defect` are recorded.
> But unlike the L5 study — where `defect` is a peer regime — the L5L6 label is argmax over the
> three series terms only. A run whose flux is mostly through pinholes is still labelled by
> whichever series resistance dominates the *intact* path. So an L5L6 `metal` cluster is NOT
> the same population as an L5 `metal` cluster; do not compare the two directly.

## Why isothermal

With `temperature` varying it monopolises the variance of log10(flux). A dummy-parameter test
on varying-T L5L6 clusters put temperature at **3.4-4.3x** the noise floor and, in the metal
cluster, **nothing else above 1.0x** — the runner-up (`crack_thickness_factor`) scored 0.84x,
below a random input the model never saw. L5L6 is worse than L5 here because it spreads 27
parameters over the residual variance instead of 19 (36 and 28 when these figures were measured).

Sampling harder does not help: the floor is flat in n (0.085 at n=400, 0.092 at n=1500),
because delta's bias for an irrelevant input does not shrink with sample size. Pinning the
temperature is what lets the remaining 26 parameters compete.

Every result table carries a **`floor`** column (max delta over the dummy inputs) and
**`delta_over_floor`**. **A value at or below 1.0 is not resolved.**

> **Caveat for `theta`:** temperature does *not* dominate that metric (1.54x oxide, 0.88x
> metal), because theta is bounded [0,1] and analysed linearly. `K_eq_ref` and `H_eq` — the
> actual surface-kinetics parameters — do surface there. So theta was the one L5L6 output the
> varying-T run handled reasonably.

## Targeting

Presets are what create the study: an untargeted sweep yields essentially no surface-limited
rows. Yields shift with temperature, so each preset's yield is measured by a probe at run
time and the draw count sized from it; Phase 1 prints them.

## Sampling, regimes and reading the indices

- **Log-uniform sampling.** Every scale parameter (pre-exponentials, thicknesses, pressure,
  defect fractions, ratios, grain size) is drawn log-uniformly, so each decade of its range
  gets equal weight; activation energies and heats are drawn linearly. The cache check
  refuses scans saved under a different `SCAN_VERSION`.
- **Regime fractions are degrees of rate control**, $X_i = \partial\ln J/\partial\ln k_i$, so a
  label names the step whose speed-up would raise the flux most. They are unit-free and sum to 1.
- **The indices are regional.** A cluster is the set of draws from one preset that the model put
  in one regime. Every index is therefore conditional on that preset's ranges and on a
  selection made on the output, and describes sensitivity *within* the regime, not over the
  whole design space. A parameter the preset pins or narrows cannot score highly. Phase 4
  repeats the analysis on each preset's full scan, without the regime selection, to show what
  the selection itself does.
- **Two noise floors.** 20 dummy inputs the model never reads give a δ floor and a PAWN floor
  (`floor`, `floor_pawn`; ratios `delta_over_floor`, `pawn_over_floor`). δ's floor stays near
  0.07 however large the cluster; PAWN's falls roughly as 1/√n, which is why the clusters
  target ~5000 rows. **A ratio at or below 1.0 is not resolved.** Clearing both floors is the
  robust result.

Run with the **`mace_env`** kernel (SALib pinned to 1.5.0).

In [1]:
import os, sys, importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
# This notebook must run with its OWN folder as the working directory: that is how
# calculations.config.model_config finds this folder's study_config.py, and how
# RESULTS_DIR below resolves. Fails loudly rather than silently using another study.
from calculations.config import assert_study_is_local
print(assert_study_is_local())

# Figures from this notebook go in one folder beside it.
from calculations.figures import results_dir, save_figure
FIGS = results_dir("sensitivity_regime_L5L6")

study: fuerst_etal_2024_model_config  (Fuerst_et_al_2024/study_config.py, via folder)


study: fuerst_etal_2024_model_config  (Fuerst_et_al_2024/study_config.py, via folder)


In [2]:
import calculations.sensitivity as sens
importlib.reload(sens)
from calculations.config.model_config import (
    SUGGESTED_RANGES_LEVEL5L6, REGIME_PRESETS, DEFAULT_SWEEP_TEMPERATURES, ACTIVE_STUDY
)
from calculations.sensitivity import (
    REGIME_LABELS, REGIME_SA_METRICS, LOG_METRICS_DEFAULT, presets_without,
    level5L6_model_wrapper, run_targeted_regime_scans, load_regime_scans,
    plot_regime_exploration, givendata_sensitivity_by_regime, summarize_givendata,
    plot_givendata_results, regime_comparison_matrix, plot_regime_comparison_heatmap,
    plot_regime_geometry, SCAN_VERSION, scan_version
)

# temperature is PINNED, not sampled -> it leaves the SA input set entirely
PRESETS = presets_without(REGIME_PRESETS, 'temperature')
PARAMS  = [p for p in SUGGESTED_RANGES_LEVEL5L6 if p != 'temperature']
print(f"study: {ACTIVE_STUDY}")
print(f"{len(PARAMS)} sampled parameters (temperature pinned); regimes: {REGIME_LABELS}")

study: fuerst_etal_2024_model_config
26 sampled parameters (temperature pinned); regimes: ('surface', 'oxide', 'metal')


In [3]:
# === Run configuration ========================================================
SEED         = 42
TEMPERATURES = list(DEFAULT_SWEEP_TEMPERATURES)      # 773 / 1073 / 1273 K
METRICS      = REGIME_SA_METRICS                     # ['flux', 'permeability', 'theta']
TOP_K        = 8

# Draw counts are no longer precomputed from a cached yield table.
# Passing N_per_regime=None makes run_targeted_regime_scans probe each
# preset and size the run from the yield it actually observes, which
# stays correct if a range, a preset or the active study changes.
N_BY_T = {T: None for T in TEMPERATURES}

RESULTS_DIR     = "sa_results"
FORCE_RECOMPUTE = False
os.makedirs(RESULTS_DIR, exist_ok=True)


## Phase 1 — Targeted per-regime scans, one set per temperature

Cached to `sa_results/scans/T<K>/scan_<regime>.csv`. The cache is only reused when every file
holds exactly the draw count currently requested — existence alone would silently serve a
previous configuration.

In [4]:
def cache_is_valid(paths, n_expected):
    """Reusable only if every scan exists, and matches the count if one was asked for.

    With auto-sizing `n_expected` is None: the draw count is decided by the probe
    at run time, so existence is all that can be checked. Set FORCE_RECOMPUTE to
    re-probe and re-scan.
    """
    d = os.path.dirname(next(iter(paths.values())))
    if scan_version(d) != SCAN_VERSION:
        return False, (f"scans in {d} are version {scan_version(d)}, not {SCAN_VERSION}: "
                       f"drawn under an older sampling or labelling design")
    for r, p in paths.items():
        if not os.path.exists(p):
            return False, f"{os.path.basename(p)} missing"
        if n_expected is None:
            continue
        n_have = len(pd.read_csv(p, usecols=[0]))
        if n_have != n_expected[r]:
            return False, (f"{os.path.basename(p)} holds {n_have} draws but config "
                           f"asks for {n_expected[r]}")
    return True, "every scan matches the requested draw counts"


clusters_T, scans_T, yields_T = {}, {}, {}

for T in TEMPERATURES:
    scan_dir = os.path.join(RESULTS_DIR, "scans", f"T{int(T)}")
    os.makedirs(scan_dir, exist_ok=True)
    paths = {r: os.path.join(scan_dir, f"scan_{r}.csv") for r in REGIME_LABELS}
    reusable, why = cache_is_valid(paths, N_BY_T[T])

    print(f"\n{'='*70}\nT = {T:.0f} K\n{'='*70}")
    if (not FORCE_RECOMPUTE) and reusable:
        print(f"  loading cached scans ({why})")
        cl, sc, _ = load_regime_scans(scan_dir, regimes=REGIME_LABELS)
        yl = {r: len(cl[r]) / len(sc[r]) for r in cl}
    else:
        print("  recomputing —", "FORCE_RECOMPUTE is set" if FORCE_RECOMPUTE else why)
        cl, sc, _, yl = run_targeted_regime_scans(
            N_per_regime=N_BY_T[T], regimes=REGIME_LABELS,
            presets=PRESETS, wrapper=level5L6_model_wrapper,
            fixed_params={'temperature': T},
            seed=SEED, save_dir=scan_dir, verbose=False)
    clusters_T[T], scans_T[T], yields_T[T] = cl, sc, yl
    print("  clusters:", {r: len(cl[r]) for r in REGIME_LABELS})
    print("  yields:  ", {r: f"{yl[r]*100:.1f}%" for r in REGIME_LABELS})

master = pd.concat(
    [pd.concat(clusters_T[T].values()).assign(temperature_K=T) for T in TEMPERATURES],
    ignore_index=True)
master.to_csv(os.path.join(RESULTS_DIR, "master_clusters.csv"), index=False)
print(f"\nmaster_clusters.csv: {len(master)} rows")


T = 773 K
  recomputing — scans in sa_results/scans/T773 are version None, not 2: drawn under an older sampling or labelling design


  saved scan -> sa_results/scans/T773/scan_surface.csv


  saved scan -> sa_results/scans/T773/scan_oxide.csv


  saved scan -> sa_results/scans/T773/scan_metal.csv
  clusters: {'surface': 5730, 'oxide': 5937, 'metal': 5666}
  yields:   {'surface': '86.5%', 'oxide': '64.4%', 'metal': '39.0%'}

T = 1073 K
  recomputing — scans in sa_results/scans/T1073 are version None, not 2: drawn under an older sampling or labelling design


  saved scan -> sa_results/scans/T1073/scan_surface.csv


  saved scan -> sa_results/scans/T1073/scan_oxide.csv


  saved scan -> sa_results/scans/T1073/scan_metal.csv
  clusters: {'surface': 5690, 'oxide': 5651, 'metal': 5690}
  yields:   {'surface': '92.6%', 'oxide': '86.1%', 'metal': '40.0%'}

T = 1273 K
  recomputing — scans in sa_results/scans/T1273 are version None, not 2: drawn under an older sampling or labelling design


  saved scan -> sa_results/scans/T1273/scan_surface.csv


  saved scan -> sa_results/scans/T1273/scan_oxide.csv


  saved scan -> sa_results/scans/T1273/scan_metal.csv
  clusters: {'surface': 5770, 'oxide': 5770, 'metal': 5616}
  yields:   {'surface': '93.5%', 'oxide': '87.5%', 'metal': '40.2%'}



master_clusters.csv: 51520 rows


In [5]:
# Median outputs by temperature and regime — the physical sanity check
print(master.groupby(['temperature_K', 'regime'])[METRICS].median().to_string())

                               flux  permeability         theta
temperature_K regime                                           
773.0         metal    3.653692e-10  2.582846e-14  2.678555e-02
              oxide    2.753321e-10  8.078999e-17  6.507214e-01
              surface  2.337335e-15  8.632239e-17  1.816391e-07
1073.0        metal    5.097494e-08  2.040611e-12  7.051542e-02
              oxide    3.014487e-07  8.785217e-14  7.694400e-01
              surface  6.620705e-14  1.974014e-15  2.059430e-08
1273.0        metal    2.706745e-07  1.002645e-11  1.026650e-01
              oxide    5.064283e-06  1.395644e-12  7.997255e-01
              surface  2.438704e-13  6.599489e-15  1.042495e-08


In [6]:
for T in TEMPERATURES:
    print(f"\n===== T = {T:.0f} K =====")
    _fig = plot_regime_exploration(clusters_T[T], output_metrics=tuple(METRICS),
                            scan_df=pd.concat(scans_T[T].values(), ignore_index=True),
                            cross_col='system_rate_limiting')
    save_figure(_fig, f"regime_exploration_{T:.0f}K", FIGS)


===== T = 773 K =====


findfont: Failed to find font weight bold for STIXSizeFiveSym, now using 400.


findfont: Failed to find font weight bold for cmmi10, now using 400.


findfont: Failed to find font weight bold for cmsy10, now using 400.


findfont: Failed to find font weight bold for cmr10, now using 400.


findfont: Failed to find font weight bold for cmtt10, now using 400.


... (6 outputs left out)


===== T = 1073 K =====



  regime (argmax)  ×  system_rate_limiting (solver):
system_rate_limiting  metal  mixed  oxide  surface
regime                                            
metal                  6081    799     28       11
oxide                     0   1134   7261        1
surface                  43   1868     19     9696



===== T = 1273 K =====



  regime (argmax)  ×  system_rate_limiting (solver):
system_rate_limiting  metal  mixed  oxide  surface
regime                                            
metal                  5982    736     18       12
oxide                     0    934   6902        0
surface                  41   1867     17    10213


## Phase 2 — Route B: given-data PAWN + delta, per temperature

Zero extra model evaluations. Each `(regime, metric)` entry carries the dummy-parameter noise
`floor`; `delta_over_floor <= 1` means the parameter is **not resolved**.

In [7]:
gd_T, tidy_T = {}, {}
for T in TEMPERATURES:
    print(f"\n===== T = {T:.0f} K =====")
    gd_T[T] = givendata_sensitivity_by_regime(
        clusters_T[T], PARAMS, output_metrics=METRICS, methods=('pawn', 'delta'),
        seed=SEED, log_metrics=LOG_METRICS_DEFAULT, n_dummy=20, verbose=True)
    tidy_T[T] = summarize_givendata(gd_T[T], PARAMS).assign(temperature_K=T)

tidy = pd.concat(tidy_T.values(), ignore_index=True)
tidy.to_csv(os.path.join(RESULTS_DIR, "routeB_givendata.csv"), index=False)
print(f"\nrouteB_givendata.csv: {len(tidy)} rows")


===== T = 773 K =====


  [surface/flux] n= 5730  floor δ=0.076 PAWN=0.041  top-δ: P_upstream (0.577)
  [surface/permeability] n= 5730  floor δ=0.076 PAWN=0.044  top-δ: P_upstream (0.309)
  [surface/theta] n= 5730  floor δ=0.273 PAWN=0.042  top-δ: P_upstream (50.631)
  [oxide/flux] n= 5937  floor δ=0.068 PAWN=0.039  top-δ: oxide_thickness (0.234)
  [oxide/permeability] n= 5937  floor δ=0.064 PAWN=0.038  top-δ: oxide_thickness (0.242)
  [oxide/theta] n= 5937  floor δ=0.068 PAWN=0.040  top-δ: K_eq_ref (0.322)
  [metal/flux] n= 5666  floor δ=0.072 PAWN=0.042  top-δ: P_upstream (0.459)
  [metal/permeability] n= 5666  floor δ=0.068 PAWN=0.052  top-δ: D_ref (0.229)
  [metal/theta] n= 5666  floor δ=0.102 PAWN=0.038  top-δ: P_upstream (12.674)

===== T = 1073 K =====


  [surface/flux] n= 5690  floor δ=0.068 PAWN=0.038  top-δ: P_upstream (0.567)
  [surface/permeability] n= 5690  floor δ=0.071 PAWN=0.038  top-δ: P_upstream (0.311)
  [surface/theta] n= 5690  floor δ=0.430 PAWN=0.037  top-δ: P_upstream (408.161)
  [oxide/flux] n= 5651  floor δ=0.066 PAWN=0.041  top-δ: H_sol_ox (0.393)
  [oxide/permeability] n= 5651  floor δ=0.072 PAWN=0.041  top-δ: H_sol_ox (0.425)
  [oxide/theta] n= 5651  floor δ=0.078 PAWN=0.038  top-δ: K_eq_ref (0.334)
  [metal/flux] n= 5690  floor δ=0.067 PAWN=0.044  top-δ: P_upstream (0.475)
  [metal/permeability] n= 5690  floor δ=0.066 PAWN=0.041  top-δ: D_ref (0.244)
  [metal/theta] n= 5690  floor δ=0.097 PAWN=0.037  top-δ: P_upstream (6.841)

===== T = 1273 K =====


  [surface/flux] n= 5770  floor δ=0.067 PAWN=0.042  top-δ: P_upstream (0.584)
  [surface/permeability] n= 5770  floor δ=0.070 PAWN=0.041  top-δ: P_upstream (0.341)
  [surface/theta] n= 5770  floor δ=0.283 PAWN=0.038  top-δ: P_upstream (3025.847)
  [oxide/flux] n= 5770  floor δ=0.067 PAWN=0.038  top-δ: H_sol_ox (0.468)
  [oxide/permeability] n= 5770  floor δ=0.070 PAWN=0.039  top-δ: H_sol_ox (0.491)
  [oxide/theta] n= 5770  floor δ=0.079 PAWN=0.040  top-δ: K_eq_ref (0.331)
  [metal/flux] n= 5616  floor δ=0.070 PAWN=0.037  top-δ: P_upstream (0.505)
  [metal/permeability] n= 5616  floor δ=0.080 PAWN=0.041  top-δ: D_ref (0.257)
  [metal/theta] n= 5616  floor δ=0.091 PAWN=0.042  top-δ: P_upstream (9.701)

routeB_givendata.csv: 702 rows


In [8]:
# Drivers of log10(flux) -- EVERY parameter, no threshold. Two noise floors: the best
# delta and the best PAWN median scored by 20 dummy inputs the model never saw. A ratio
# at or below 1.0 is indistinguishable from noise. delta's floor barely shrinks with
# cluster size; PAWN's does, so at these cluster sizes PAWN resolves smaller effects.
for T in TEMPERATURES:
    for reg in REGIME_LABELS:
        sub = (tidy[(tidy.metric == 'flux') & (tidy.temperature_K == T)
                    & (tidy.regime == reg)]
               .sort_values('delta_over_floor', ascending=False))
        if sub.empty:
            continue
        fd, fp = sub['floor'].iloc[0], sub['floor_pawn'].iloc[0]
        res = (sub.delta_over_floor > 1.0) | (sub.pawn_over_floor > 1.0)
        print(f"  T={T:.0f}K  {reg:8s} n={sub.n.iloc[0]}  floor: delta={fd:.3f} "
              f"PAWN={fp:.3f}  ({int(res.sum())}/{len(sub)} above a floor)")
        for _, r_ in sub.head(8).iterrows():
            ok = r_.delta_over_floor > 1.0 or r_.pawn_over_floor > 1.0
            print(f"        {r_.parameter:24s} delta={r_.delta:.3f} ({r_.delta_over_floor:.2f}x)"
                  f"  PAWN={r_.pawn_median:.3f} ({r_.pawn_over_floor:.2f}x)"
                  + ('' if ok else '   <- at/below noise'))
    print()

  T=773K  surface  n=5730  floor: delta=0.076 PAWN=0.041  (8/26 above a floor)
        P_upstream               delta=0.577 (7.64x)  PAWN=0.518 (12.51x)
        k_diss_ref               delta=0.082 (1.09x)  PAWN=0.087 (2.10x)
        D_ref                    delta=0.073 (0.97x)  PAWN=0.045 (1.09x)
        f_pinhole                delta=0.069 (0.92x)  PAWN=0.067 (1.61x)
        oxide_thickness          delta=0.068 (0.89x)  PAWN=0.042 (1.02x)
        D_ox_ref                 delta=0.067 (0.89x)  PAWN=0.049 (1.18x)
        K_s_ref                  delta=0.065 (0.86x)  PAWN=0.042 (1.03x)
        K_eq_metal_ref           delta=0.064 (0.84x)  PAWN=0.027 (0.65x)   <- at/below noise
  T=773K  oxide    n=5937  floor: delta=0.068 PAWN=0.039  (12/26 above a floor)
        oxide_thickness          delta=0.234 (3.42x)  PAWN=0.234 (5.94x)
        P_upstream               delta=0.197 (2.88x)  PAWN=0.212 (5.38x)
        H_sol_ox                 delta=0.151 (2.21x)  PAWN=0.169 (4.30x)
        K_s_ref  

In [9]:
for reg in REGIME_LABELS:
    _fig = plot_givendata_results(gd_T[TEMPERATURES[-1]], reg, 'flux', PARAMS, top_n=5)
    save_figure(_fig, f"givendata_{reg}_flux", FIGS)

## Phase 2 (cont.) — regime geometry (highest temperature)

Clusters projected on their two strongest drivers. Dashed boxes are the 10-90 percentile band;
overlap is the reminder that regime boundaries are curved, not rectangular.

In [10]:
T_show = TEMPERATURES[-1]
def regime_topk(reg, k=TOP_K, metric='flux', T=None):
    """Top-k parameters for one regime at one temperature, ranked by Route-B delta."""
    T = T_show if T is None else T
    sub = tidy[(tidy.temperature_K == T) & (tidy.regime == reg) & (tidy.metric == metric)]
    return sub.nlargest(k, 'delta')['parameter'].tolist()

for reg in REGIME_LABELS:
    _fig = plot_regime_geometry(clusters_T[T_show], axes=regime_topk(reg)[:2],
                         log_x=True, log_y=True,
                         title=f"L5L6 regime geometry @ {T_show:.0f} K — top-2 delta drivers of '{reg}'")
    save_figure(_fig, f"regime_geometry_{reg}", FIGS)

## Phase 3 — Cross-regime comparison, per temperature

Columns are column-normalised so each regime spans a full 0-1 scale; the small corner number is
the raw index. Exported `compare_*.csv` files are always raw.

In [11]:
for T in TEMPERATURES:
    print(f"\n===== T = {T:.0f} K — delta, log10(flux) =====")
    _fig = plot_regime_comparison_heatmap(gd_T[T], 'flux', PARAMS, index='delta', top_union=5)
    save_figure(_fig, f"comparison_heatmap_{T:.0f}K", FIGS)
    for idx in ('delta', 'pawn_median'):
        for met in METRICS:
            mat = regime_comparison_matrix(gd_T[T], met, PARAMS, index=idx, top_union=5)
            if not mat.empty:
                mat.to_csv(os.path.join(RESULTS_DIR, f"compare_{idx}_{met}_T{int(T)}.csv"))


===== T = 773 K — delta, log10(flux) =====



===== T = 1073 K — delta, log10(flux) =====



===== T = 1273 K — delta, log10(flux) =====


In [12]:
# How does each parameter's margin over the noise floor move with temperature?
# Unfiltered: every parameter appears at every T, so a value near 1.0 reads as
# "at the noise floor" instead of vanishing into a NaN that looks like missing data.
piv = (tidy[tidy.metric == 'flux']
       .pivot_table(index=['regime', 'parameter'], columns='temperature_K',
                    values='delta_over_floor'))
piv.columns = [f"{c:.0f}K" for c in piv.columns]
piv['best'] = piv.max(axis=1)

print("delta_over_floor for log10(flux) — ALL parameters, no threshold")
print("  1.0 = exactly at the dummy-parameter noise floor; higher = resolved\n")
for reg in REGIME_LABELS:
    fl = (tidy[(tidy.metric == 'flux') & (tidy.regime == reg)]
          .groupby('temperature_K')['floor'].first())
    print(f"[{reg}]  floor: " + "  ".join(f"{T:.0f}K={v:.3f}" for T, v in fl.items()))
    print(piv.loc[reg].sort_values('best', ascending=False).round(2).head(12).to_string())
    print()

piv.to_csv(os.path.join(RESULTS_DIR, "delta_over_floor_by_T.csv"))
print("saved delta_over_floor_by_T.csv")

delta_over_floor for log10(flux) — ALL parameters, no threshold
  1.0 = exactly at the dummy-parameter noise floor; higher = resolved

[surface]  floor: 773K=0.076  1073K=0.068  1273K=0.067
                  773K  1073K  1273K  best
parameter                                 
P_upstream        7.64   8.36   8.65  8.65
k_diss_ref        1.09   1.27   1.23  1.27
k_diss_metal_ref  0.68   0.85   1.08  1.08
f_pinhole         0.92   1.01   1.01  1.01
E_diss            0.69   1.00   0.92  1.00
H_eq              0.70   0.97   0.94  0.97
D_ref             0.97   0.81   0.86  0.97
H_eq_metal        0.83   0.94   0.86  0.94
E_D               0.62   0.76   0.92  0.92
K_eq_ref          0.74   0.87   0.92  0.92
H_sol_ox          0.68   0.86   0.92  0.92
metal_thickness   0.75   0.91   0.75  0.91

[oxide]  floor: 773K=0.068  1073K=0.066  1273K=0.067
                  773K  1073K  1273K  best
parameter                                 
H_sol_ox          2.21   5.94   6.97  6.97
oxide_thickness   3.42   

In [13]:
# theta is the metric temperature does NOT dominate — worth its own look
piv_t = (tidy[tidy.metric == 'theta']
         .pivot_table(index=['regime', 'parameter'], columns='temperature_K',
                      values='delta_over_floor'))
piv_t.columns = [f"{c:.0f}K" for c in piv_t.columns]
piv_t['best'] = piv_t.max(axis=1)
print("delta_over_floor for theta (surface coverage) — top 8 per regime\n")
for reg in REGIME_LABELS:
    print(f"[{reg}]")
    print(piv_t.loc[reg].sort_values('best', ascending=False).round(2).head(8).to_string())
    print()

delta_over_floor for theta (surface coverage) — top 8 per regime

[surface]
                    773K   1073K     1273K      best
parameter                                           
P_upstream        185.32  948.64  10696.91  10696.91
K_eq_ref            3.91    1.60      2.62      3.91
k_diss_ref          0.69    0.29      1.62      1.62
K_s_ref             0.46    0.30      1.34      1.34
f_crack             0.55    0.43      1.25      1.25
gb_thickness        0.53    0.48      1.04      1.04
k_diss_metal_ref    1.01    0.51      0.84      1.01
f_gb_defect         0.42    0.39      0.99      0.99

[oxide]
                 773K  1073K  1273K  best
parameter                                
K_eq_ref         4.77   4.31   4.17  4.77
P_upstream       2.53   2.48   2.29  2.53
H_eq             1.30   0.73   0.90  1.30
H_eq_metal       1.04   0.85   0.83  1.04
f_pinhole        1.03   0.79   0.89  1.03
D_ref            0.96   0.92   0.79  0.96
K_ox_ref         0.94   0.88   0.82  0.94
metal_t

## Phase 4 — Regional versus preset-wide indices

The clusters are selected on the output, so the indices above are regional (see the top of
the notebook). Here the same estimators run on each preset's **full** scan — every draw,
whatever regime it landed in, subsampled to the cluster size — which removes the selection
but keeps the preset. Where a parameter ranks very differently in the two, it is the regime
selection, not the input on its own, that is doing the work.

In [14]:
N_PW = 5000   # subsample the full scans to about the cluster size, so floors compare
pw_T = {}
for T in TEMPERATURES:
    full = {r: df.sample(min(len(df), N_PW), random_state=SEED).reset_index(drop=True)
            for r, df in scans_T[T].items()}
    gd_pw = givendata_sensitivity_by_regime(
        full, PARAMS, output_metrics=['flux'], methods=('pawn', 'delta'),
        seed=SEED, log_metrics=LOG_METRICS_DEFAULT, n_dummy=20, verbose=False)
    pw_T[T] = summarize_givendata(gd_pw, PARAMS).assign(temperature_K=T)
pw = pd.concat(pw_T.values(), ignore_index=True)
pw.to_csv(os.path.join(RESULTS_DIR, "routeB_presetwide.csv"), index=False)

key = ['temperature_K', 'regime', 'parameter']
cols = ['delta_over_floor', 'pawn_over_floor']
both = (tidy[tidy.metric == 'flux'].set_index(key)[cols]
        .join(pw.set_index(key)[cols], rsuffix='_preset'))
for T in TEMPERATURES:
    for reg in REGIME_LABELS:
        print(f"\n[{reg} @ {T:.0f} K]  in-regime cluster vs whole '{reg}' preset scan "
              f"(log10 flux, x floor)")
        print(both.loc[(T, reg)].sort_values('delta_over_floor', ascending=False)
              .head(6).round(2).to_string())


[surface @ 773 K]  in-regime cluster vs whole 'surface' preset scan (log10 flux, x floor)
                 delta_over_floor  pawn_over_floor  delta_over_floor_preset  pawn_over_floor_preset
parameter                                                                                          
P_upstream                   7.64            12.51                     8.40                   12.17
k_diss_ref                   1.09             2.10                     1.28                    2.16
D_ref                        0.97             1.09                     0.87                    1.00
f_pinhole                    0.92             1.61                     1.00                    1.43
oxide_thickness              0.89             1.02                     0.83                    0.76
D_ox_ref                     0.89             1.18                     0.96                    0.97

[oxide @ 773 K]  in-regime cluster vs whole 'oxide' preset scan (log10 flux, x floor)
                 delta

In [15]:
# Manifest of everything saved
import glob
print("Saved CSV files under", os.path.abspath(RESULTS_DIR), ":")
for f in sorted(glob.glob(os.path.join(RESULTS_DIR, "**", "*.csv"), recursive=True)):
    print("  ", os.path.relpath(f, RESULTS_DIR), f"({os.path.getsize(f)//1024} KB)")

Saved CSV files under /Users/rwest/CoMoChEng Dropbox/Richard West/Northeastern/Research/Research-MHI-2026-hydrogen/Continuum_H_permeation/Fuerst_et_al_2024/sa_results :
   compare_delta_flux_T1073.csv (0 KB)
   compare_delta_flux_T1273.csv (0 KB)
   compare_delta_flux_T773.csv (0 KB)
   compare_delta_permeability_T1073.csv (0 KB)
   compare_delta_permeability_T1273.csv (0 KB)
   compare_delta_permeability_T773.csv (0 KB)
   compare_delta_theta_T1073.csv (0 KB)
   compare_delta_theta_T1273.csv (0 KB)
   compare_delta_theta_T773.csv (0 KB)
   compare_pawn_median_flux_T1073.csv (0 KB)
   compare_pawn_median_flux_T1273.csv (0 KB)
   compare_pawn_median_flux_T773.csv (0 KB)
   compare_pawn_median_permeability_T1073.csv (0 KB)
   compare_pawn_median_permeability_T1273.csv (0 KB)
   compare_pawn_median_permeability_T773.csv (0 KB)
   compare_pawn_median_theta_T1073.csv (0 KB)
   compare_pawn_median_theta_T1273.csv (0 KB)
   compare_pawn_median_theta_T773.csv (0 KB)
   delta_over_floor_by_T.cs